# The Gaussian Elimination Algorithm

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "The Gaussian Elimination Algorithm" (DeepLearning.AI).*

This pulls every Week-2 idea — augmented matrices, row operations, REF, RREF, back-substitution — into one **algorithm** that solves any linear system.

**Gaussian elimination — summary**
1. Build the **augmented matrix** $[A \mid \mathbf{b}]$.
2. Reduce it to **reduced row echelon form** (forward elimination, scaling pivots to 1, clearing below).
3. Complete **back-substitution** (clear above the pivots).
4. **Stop if you hit a row of 0s** — that flags infinitely many or no solutions.

In [1]:
import numpy as np
from fractions import Fraction as F

def show(M):
    for row in M:
        *coef, rhs = row
        print("   [" + "  ".join(f"{str(x):>5}" for x in coef) + f"  |  {str(rhs):>4}]")

## 1. Worked example

$$ \begin{cases} 2a - b + c = 1 \\ 2a + 2b + 4c = -2 \\ 4a + b = -1 \end{cases} \;\longrightarrow\; \left[\begin{array}{ccc|c} 2 & -1 & 1 & 1 \\ 2 & 2 & 4 & -2 \\ 4 & 1 & 0 & -1 \end{array}\right] $$

**Pivoting on row 1** — divide $R_1$ by 2, then clear column 1 below it ($R_2 \leftarrow R_2 - 2R_1$, $R_3 \leftarrow R_3 - 4R_1$):

$$ \left[\begin{array}{ccc|c} 1 & -\tfrac12 & \tfrac12 & \tfrac12 \\ 0 & 3 & 3 & -3 \\ 0 & 3 & -2 & -3 \end{array}\right] $$

**Pivoting on row 2** — divide $R_2$ by 3, clear below ($R_3 \leftarrow R_3 - 3R_2$), then divide $R_3$ by $-5$:

$$ \left[\begin{array}{ccc|c} 1 & -\tfrac12 & \tfrac12 & \tfrac12 \\ 0 & 1 & 1 & -1 \\ 0 & 0 & 1 & 0 \end{array}\right] \;\Rightarrow\; c = 0. $$

**Back-substitution** (clear above the pivots) gives the RREF and the answer $a = 0,\; b = -1,\; c = 0$.

In [2]:
def gaussian_eliminate(aug, verbose=False):
    """Reduce an augmented matrix [A | b] to RREF (exact). Returns (rref, pivot_cols)."""
    M = [[F(x) for x in row] for row in aug]
    rows, cols = len(M), len(M[0])
    ncoef = cols - 1                       # last column is the constants
    pivots, prow = [], 0
    # forward elimination -> REF
    for col in range(ncoef):
        piv = next((r for r in range(prow, rows) if M[r][col] != 0), None)
        if piv is None:
            continue
        M[prow], M[piv] = M[piv], M[prow]
        M[prow] = [x / M[prow][col] for x in M[prow]]
        for r in range(prow + 1, rows):
            M[r] = [M[r][k] - M[r][col] * M[prow][k] for k in range(cols)]
        pivots.append(col); prow += 1
        if verbose:
            print(f"pivot on column {col}:"); show(M); print()
    # back-substitution -> RREF
    for prow, col in enumerate(pivots):
        for r in range(prow):
            M[r] = [M[r][k] - M[r][col] * M[prow][k] for k in range(cols)]
    return M, pivots

aug = [[2, -1, 1, 1], [2, 2, 4, -2], [4, 1, 0, -1]]
rref, pivots = gaussian_eliminate(aug, verbose=True)
print("RREF:"); show(rref)
print("\nSolution: a =", rref[0][-1], ", b =", rref[1][-1], ", c =", rref[2][-1])

pivot on column 0:
   [    1   -1/2    1/2  |   1/2]
   [    0      3      3  |    -3]
   [    0      3     -2  |    -3]

pivot on column 1:
   [    1   -1/2    1/2  |   1/2]
   [    0      1      1  |    -1]
   [    0      0     -5  |     0]

pivot on column 2:
   [    1   -1/2    1/2  |   1/2]
   [    0      1      1  |    -1]
   [    0      0      1  |     0]

RREF:
   [    1      0      0  |     0]
   [    0      1      0  |    -1]
   [    0      0      1  |     0]

Solution: a = 0 , b = -1 , c = 0


In [3]:
# Cross-check with numpy.
A = np.array([[2.0, -1, 1], [2, 2, 4], [4, 1, 0]])
b = np.array([1.0, -2, -1])
print("numpy solution [a, b, c] =", np.linalg.solve(A, b))

numpy solution [a, b, c] = [ 0. -1.  0.]


## 2. Step 4: stopping at a row of zeros

If elimination produces a row whose **coefficients are all 0**, the matrix is singular — the system has either infinitely many or no solutions. The constant on that row decides which:

- $[\,0\;0\;0 \mid 0\,]$ → consistent → **infinitely many** solutions,
- $[\,0\;0\;0 \mid k\neq 0\,]$ → contradiction → **no** solution.

For example $\left[\begin{array}{ccc|c}1&2&-1&5\\2&4&5&1\\3&6&4&6\end{array}\right]$ row-reduces to a form with a bottom **row of zeros**:

In [4]:
def classify(aug):
    M = [[F(x) for x in row] for row in aug]
    rref, pivots = gaussian_eliminate(aug)
    ncoef = len(aug[0]) - 1
    n_unknowns = ncoef
    # a zero coefficient-row with non-zero constant => contradiction
    for row in rref:
        if all(x == 0 for x in row[:ncoef]) and row[-1] != 0:
            return "no solution (contradictory)"
    if len(pivots) < n_unknowns:
        return "infinitely many solutions (redundant)"
    return "unique solution"

example = [[1, 2, -1, 5], [2, 4, 5, 1], [3, 6, 4, 6]]
rref, pivots = gaussian_eliminate(example)
print("RREF:"); show(rref)
print("\npivots:", len(pivots), "of", len(example[0]) - 1, "unknowns")
print("->", classify(example))

RREF:
   [    1      2      0  |  26/7]
   [    0      0      1  |  -9/7]
   [    0      0      0  |     0]

pivots: 2 of 3 unknowns
-> infinitely many solutions (redundant)


In [5]:
# All three outcomes from the same coefficient matrix, different constants.
print("unique     :", classify([[2, -1, 1, 1], [2, 2, 4, -2], [4, 1, 0, -1]]))
print("redundant  :", classify([[1, 1, 10], [2, 2, 20]]))      # 0 = 0 row
print("contradict.:", classify([[1, 1, 10], [2, 2, 24]]))      # 0 = k row

unique     : unique solution
redundant  : infinitely many solutions (redundant)
contradict.: no solution (contradictory)


## 3. Takeaways

- **Gaussian elimination** is the complete recipe to solve a linear system:
  1. form the **augmented matrix** $[A\mid b]$,
  2. **forward-eliminate** to REF (pivot, scale to 1, clear below),
  3. **back-substitute** to RREF (clear above) and read off the solution,
  4. **watch for a zero row** — it signals a singular system.
- A zero coefficient-row with constant $0$ ⇒ **infinitely many** solutions; with a non-zero constant ⇒ **no** solution.
- This is the algorithm numerical libraries implement (as **LU decomposition**) inside `numpy.linalg.solve`.

*Companion:* [reduced row echelon form](./C1_W2_L2_reduced_row_echelon_form.ipynb) · [solving singular systems](./C1_W2_L1_solving_singular_systems.ipynb) (infinite vs. none).